In [16]:
import pandas as pd
import re
import nltk
import pickle

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# -------------------------
# Load dataset
# -------------------------
df = pd.read_csv("../data/Resume.csv")
df.columns = df.columns.str.strip()

# print("Original shape:", df.shape)
# print("Available Categories:\n", df['Category'].unique())

print("Before filter:", df.shape)
tech_roles = [
    "INFORMATION-TECHNOLOGY",
    "ENGINEERING"
]

df = df[df['Category'].isin(tech_roles)]

print("After filter:", df.shape)
print(df['Category'].value_counts())

# -------------------------
# IMPORTANT: Comment this first (to avoid empty data)
# -------------------------
# tech_roles = [...]
# df = df[df['Category'].isin(tech_roles)]

# -------------------------
# Clean text
# -------------------------
def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = re.sub(r'[^a-zA-Z]', ' ', str(text))
    text = text.lower()
    words = text.split()
    
    return " ".join(words)

df['cleaned_resume'] = df['Resume_str'].apply(clean_text)

# Remove empty rows
df = df[df['cleaned_resume'].str.strip() != ""]

print("After cleaning:", df.shape)

# -------------------------
# TF-IDF
# -------------------------
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), stop_words='english')
X = tfidf.fit_transform(df['cleaned_resume'])

# -------------------------
# Labels
# -------------------------
le = LabelEncoder()
y = le.fit_transform(df['Category'])

# -------------------------
# Train
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearSVC()
model.fit(X_train, y_train)

# -------------------------
# Accuracy
# -------------------------
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

# -------------------------
# Save
# -------------------------
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))
pickle.dump(le, open("label_encoder.pkl", "wb"))

print("✅ DONE - Model saved")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Before filter: (2484, 4)
After filter: (238, 4)
Category
INFORMATION-TECHNOLOGY    120
ENGINEERING               118
Name: count, dtype: int64
After cleaning: (238, 5)
Accuracy: 0.9583333333333334
✅ DONE - Model saved
